# Soil 16S Phylogeny Pilot

## Species finding and relatedness from a class-safe cache

Run the cells from top to bottom. When you see a **Think** prompt, pause and write one sentence.

Today you will:

1. load cached 16S rRNA reference sequences from soil-relevant bacteria,
2. compare two unknown soil ASV/read sequences with those references,
3. inspect a small aligned marker window,
4. turn sequence differences into a distance matrix,
5. build UPGMA and neighbor-joining trees,
6. compare this controlled tree exercise with real Atacama soil ASV abundance patterns,
7. report a careful closest-reference claim.

A tree is a hypothesis from evidence. In this notebook the evidence is one short 16S marker window, so the final claim must stay cautious.

This is a browser-only 16S marker/metabarcoding pilot. It is not a shotgun metagenomics pipeline and it does not prove exact species identity.


In [ ]:
#@title Class controls { display-mode: "form" }
USE_GITHUB_CACHE = False #@param {type:"boolean"}
CACHE_BASE_URL = "" #@param {type:"string"}
QUERY_TO_REPORT = "Soil_ASV_A" #@param ["Soil_ASV_A", "Soil_ASV_B"]
ATACAMA_ASV_TO_PLOT = "Atacama_ASV_01" #@param ["Atacama_ASV_01", "Atacama_ASV_02", "Atacama_ASV_03", "Atacama_ASV_04", "Atacama_ASV_05", "Atacama_ASV_06", "Atacama_ASV_07", "Atacama_ASV_08", "Atacama_ASV_09", "Atacama_ASV_10", "Atacama_ASV_11", "Atacama_ASV_12"]
TREE_METHOD_TO_SHOW = "Compare UPGMA and neighbor joining" #@param ["UPGMA", "Neighbor joining", "Compare UPGMA and neighbor joining"]
MARKER_WINDOW_BASES = 520 #@param {type:"slider", min:200, max:560, step:20}
ALIGNMENT_START = 130 #@param {type:"slider", min:0, max:420, step:10}
ALIGNMENT_WIDTH = 70 #@param {type:"slider", min:30, max:100, step:10}
print("Controls set. The default path uses the embedded cache, so class runs do not depend on live BLAST.")


In [ ]:
#@title Install and import notebook dependencies { display-mode: "form" }
import importlib.util
import subprocess
import sys

def ensure(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, import_name in [
    ("biopython", "Bio"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
]:
    ensure(package, import_name)

import json
import math
import urllib.request
import xml.etree.ElementTree as ET
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

from Bio import Phylo, SeqIO
from Bio.Align import PairwiseAligner
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

EMBEDDED_CACHE = {
  "pilot_16s_references.fasta": ">Bacillus_subtilis_168 accession=NR_102783.2 source=NCBI_Nucleotide role=reference species=\"Bacillus subtilis subsp. subtilis strain 168\"\nTTATCGGAGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGGACAGATGGGAG\nCTTGCTCCCTGATGTTAGCGGCGGACGGGTGAGTAACACGTGGGTAACCTGCCTGTAAGACTGGGATAACTCCGGGAAAC\nCGGGGCTAATACCGGATGGTTGTTTGAACCGCATGGTTCAAACATAAAAGGTGGCTTCGGCTACCACTTACAGATGGACC\nCGCGGCGCATTAGCTAGTTGGTGAGGTAACGGCTCACCAAGGCGACGATGCGTAGCCGACCTGAGAGGGTGATCGGCCAC\nACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTAGGGAATCTTCCGCAATGGACGAAAGTCTGACGGAG\nCAACGCCGCGTGAGTGATGAAGGTTTTCGGATCGTAAAGCTCTGTTGTTAGGGAAGAACAAGTGCCGTTCGAATAGGGCG\nGTACCTTGACGGTACCTAACCAGAAAGCCACGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGTGGCAAGCGTTG\nTCCGGAATTATTGGGCGTAAAGGGCTCGCAGGCGGTTTCTTAAGTCTGATGTGAAAGCCCCCGGCTCAACCGGGGAGGGT\nCATTGGAAACTGGGGAACTTGAGTGCAGAAGAGGAGAGTGGAATTCCACGTGTAGCGGTGAAATGCGTAGAGATGTGGAG\nGAACACCAGTGGCGAAGGCGACTCTCTGGTCTGTAACTGACGCTGAGGAGCGAAAGCGTGGGGAGCGAACAGGATTAGAT\nACCCTGGTAGTCCACGCCGTAAACGATGAGTGCTAAGTGTTAGGGGGTTTCCGCCCCTTAGTGCTGCAGCTAACGCATTA\nAGCACTCCGCCTGGGGAGTACGGTCGCAAGACTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGT\nGGTTTAATTCGAAGCAACGCGAAGAACCTTACCAGGTCTTGACATCCTCTGACAATCCTAGAGATAGGACGTCCCCTTCG\nGGGGCAGAGTGACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAAC\nCCTTGATCTTAGTTGCCAGCATTCAGTTGGGCACTCTAAGGTGACTGCCGGTGACAAACCGGAGGAAGGTGGGGATGACG\nTCAAATCATCATGCCCCTTATGACCTGGGCTACACACGTGCTACAATGGACAGAACAAAGGGCAGCGAAACCGCGAGGTT\nAAGCCAATCCCACAAATCTGTTCTCAGTTCGGATCGCAGTCTGCAACTCGACTGCGTGAAGCTGGAATCGCTAGTAATCG\nCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCACGAGAGTTTGTAACACCCGA\nAGTCGGTGAGGTAACCTTTTAGGAGCCAGCCGCCGAAGGTGGGACAGATGATTGGGGTGAAGTCGTAACAAGGTAGCCGT\nATCGGAAGGTGCGGCTGGATCACCTCCTTT\n>Pseudomonas_fluorescens_CCM2115 accession=NR_115715.1 source=NCBI_Nucleotide role=reference species=\"Pseudomonas fluorescens strain CCM 2115\"\nAGAGTTTGATCCTGGCTCAGATTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAGCGGTAGAGAGAAGCTTGCTTC\nTCTTGAGAGCGGCGGACGGGTGAGTAAAGCCTAGGAATCTGCCTGGTAGTGGGGGATAACGTTCGGAAACGGACGCTAAT\nACCGCATACGTCCTACGGGAGAAAGCAGGGGACCTTCGGGCCTTGCGCTATCAGATGAGCCTAGGTCGGATTAGCTAGTT\nGGTGAGGTAATGGCTCACCAAGGCGACGATCCGTAACTGGTCTGAGAGGATGATCAGTCACACTGGAACTGAGACACGGT\nCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGGACAATGGGCGAAAGCCTGATCCAGCCATGCCGCGTGTGTGAAG\nAAGGTCTTCGGATTGTAAAGCACTTTAAGTTGGGAGGAAGGGCATTAACCTAATACGTTAGTGTTTTGACGTTACCGACA\nGAATAAGCACCGGCTAACTCTGTGCCAGCAGCCGCGGTAATACAGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAA\nAGCGCGCGTAGGTGGTTTGTTAAGTTGGATGTGAAATCCCCGGGCTCAACCTGGGAACTGCATTCAAAACTGACTGACTA\nGAGTATGGTAGAGGGTGGTGGAATTTCCTGTGTAGCGGTGAAATGCGTAGATATAGGAAGGAACACCAGTGGCGAAGGCG\nACCACCTGGACTAATACTGACACTGAGGTGCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCACGCCGT\nAAACGATGTCAACTAGCCGTTGGGAGCCTTGAGCTCTTAGTGGCGCAGCTAACGCATTAAGTTGACCGCCTGGGGAGTAC\nGGCCGCAAGGTTAAAACTCAAATGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGAAGCAACGCG\nAAGAACCTTACCAGGCCTTGACATCCAATGAACTTTCTAGAGATAGATTGGTGCCTTCGGGAACATTGAGACAGGTGCTG\nCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGTAACGAGCGCAACCCTTGTCCTTAGTTACCAGCA\nCGTAATGGTGGGCACTCTAAGGAGACTGCCGGTGACAAACCGGAGGAAGGTGGGGATGACGTCAAGTCATCATGGCCCTT\nACGGCCTGGGCTACACACGTGCTACAATGGTCGGTACAGAGGGTTGCCAAGCCGCGAGGTGGAGCTAATCCCACAAAACC\nGATCGTAGTCCGGATCGCAGTCTGCAACTCGACTGCGTGAAGTCGGAATCGCTAGTAATCGCGAATCAGAATGTCGCGGT\nGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCATGGGAGTGGGTTGCACCAGAAGTAGCTAGTCTAACCTTC\nGGGAGGACGGTTACCACGGTGTGATTCATGACTGGGGTGAAGTCGTAACAAGGTAGCCGTAGGGGAACCTGCGGCTGGAT\n>Streptomyces_coelicolor_rrnD accession=Y00411.1 source=NCBI_Nucleotide role=reference species=\"Streptomyces coelicolor\"\nTGGGCCCGCATCACCATCGGCGTCCTCGCCGAGCTGGCCTTCCTGGCCTACGTCTACGTTCTGGGCGGCCGAGCCGTGCG\nCGACGGCGAGACGGGTGACGTCGAGGCAGCCGAACGCAGCGCCACGGTGCCAACAGCCGCCTGATGTGCATCCACCCCTG\nCGAGCTGCTAGTGTCCTCTTCGTTCCCGCAAGAGCCGTTGACACGGAGCGAGCGGGGAGGTAGATTCGAACAGTTGCCTG\nGAGACGGGTTCACCCCAGAGGGCAACAGTGAACATCTACCAGCTTCTCCGAATCAACGAATTCGACGAAGCACTCTCCCG\nATGAATCGGAAACGAAGGCCGGTAAGACCGGCTCGAAAGTTCTGATAAAGTCGGAGCCGCCGGAAAGGGAAACGCGAAAG\nCGGGAACCTGGAAAGCGCCGAGGAAATCGGATCGGAAAGATCTGATAGAGTCGGAAACGCAAGACCGAAGGGAAGCGCCC\nGGAGGAAAGCCCGAGAGGGTGAGTACAAAGGAAGCGTCCGTTCCTTGAGAACTCAACAGCGTGCCAAAAGTCAACGCCAG\nATATGTTGATACCCCGACCTGATCGGATCTCCGTTCGGGTTGAGGTTCCTTTGAAGTAACACAACAGCGAGGACGCTGTG\nAACGGTCGGATTATTCCTCCGACTGTTCCGCTCTCGTGGTGTCACCCGATTACGGGTATACATTCACGGAGAGTTTGATC\nCTGGCTCAGGACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGATGAACCACTTCGGTGGGGATTAGTGGCGA\nACGGGTGAGTAACACGTGGGCAATCTGCCCTTCACTCTGGGACAAGCCCTGGAAACGGGGTCTAATACCGGATACTGACC\nCTCGCAGGCATCTGCGAGGTTCGAAAGCTCCGGCGGTGAAGGATGAGCCCGCGGCCTATCAGCTTGTTGGTGAGGTAATG\nGCTCACCAAGGCGACGACGGGTAGCCGGCCTGAGAGGGCGACCGGCCACACTGGGACTGAGACACGGCCCAGACTCCTAC\nGGGAGGCAGCAGTGGGGAATGTTGCACAATGGGCGAAAGCCTGATGCAGCGACGCCGCGTGAGGGATGACGGCCTTCGGG\nTTGTAAACCTCTTTCAGCAGGGAAGAAGCGAAAGTGACGGTACCTGCAGAAGAAGCGCCGGCTAACTACGTGCCAGCAGC\nCGCGGTAATACGTAGGGCGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGCTTGTCACGTCGGTTGT\nGAAAGCCCGGGGCTTAACCCCGCCACTGCAGTCGATACGGGCAGGCTAGAGTTCGGTAGGGGAGATCGGAATTCCTGGTG\nTAGCGGTGAAATGCGCAGATATCAGGAGGAACACCGGTGGCGAAGGCGGATCTCTGGGCCGATACTGACGCTGAGGAGCG\nAAAGNGTGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGGTGGGCACTAGGTGTGGGCAACATTCC\nACGTTGTCCGTGCCGCAGCTAACGCATTAAGTGCCCCGCCTGGGGAGTACGGCCGCAAGGCTAAAACTCAAAGGAATTGA\nCGGGGGCCCGCACAAGCGGCGGAGCATGTGGCTTAATTCGACGCAACGCGAAGAACCTTACCAAGGCTTGACATACACCG\nGAAAGCATCAGAGATGGTGCCCCCCTTGTGGTCGGTGTACAGGTGGTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGT\nTGGGTTAAGTCCCGCAACGAGCGCAACCCTTGTCCCGTGTTGCCAGCAAGCCCTTCGGGGTGTTGGGGACTCACGGGAGA\nCCGCCGGGGTCAACTCGGAGGAAGGTGGGGACGACGTCAAGTCATCATGCCCCTTATGTCTTGGGCTGCACACGTGCTAC\nAATGGCCGGTACAATGAGCTGCGATACCGCAAGGTGGAGCGAATCTCAAAAAGCCGGTCTCAGTTCGGATTGGGGTCTGC\nAACTCGACCCCATGAAGTCGGAGTCGCTAGTAATCGCAGATCAGCATTGCTGCGGTGAATACGTTCCCGGGCCTTGTACA\nCACCGCCCGTCACGTCACGAAAGTCGGTAACACCCGAAGCCGGTGGCCCAACCCCTTGTGGGAGGGAGCTGTCGAAGGTG\nGGACTGGCGATTGGGACGAAGTCGTAACAAGGTAGCCGTACCGGAAGGTGCGGCTGGATCACCTCCTTTCTAAGGAGCAC\nATAGCCGACTGCAGCGAAATGTCCTGCACGGTTGCTCATGGGTGGAACGTTGACTACTCGGCACGGTCTTCTTGATGGAT\nCACTAGTACTGCTTCGGCGTGGAACGTGACTTCAAAGAGGGGTTCGTGTCGGGCACGCTGTTGGGTATCTGAGGGTACGG\nCCGTGAGGTCGCCTTCAGTTGCCGGCCCCGGTAAAAATCCGCGTGAGTGGGTTGTGACGGGTGGTTGGTCGTTGTTTGAG\nAACTGCACAGTGGACGCGAGCATCTGTGGCCAAGTTTTTAAGGGCGCACGGTGGATGCCTT\n>Rhizobium_leguminosarum_IAM12609 accession=D14513.1 source=NCBI_Nucleotide role=reference species=\"Rhizobium leguminosarum type strain IAM 12609\"\nAACTTGAGAGTTTGATCCTGGCTCAGAACGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCCCGCAANNNNA\nGCGGCAGACGGGTGAGTAACGCGTGGGAACGTACCCTTTACTACGGAATAACGCAGGGAAACTTGTGCTAATACCGTATG\nTGCCCTTTGGGGGAAAGATTTATCGGTAAAGGATCGGCCCGCGTTGGATTAGCTAGTTGGTGGGGTAAAGGCCTACCAAG\nGCGACGATCCATAGCTGGTCTGAGAGGATGATCAGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGC\nAGTGGGGAATATTGGACAATGGGCGCAAGCCTGATCCAGCCATGCCGCGTGAGTGATGAAGGCCCTAGGGTTGTAAAGCT\nCTTTCACCGGAGAAGATAATGACGGTATCCGGAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGGTAATACGAAG\nGGGGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGATCGATAAGTCAGGGGTGAAATCCCAGGGCTC\nAACCCTGGAACTGCCTTTGATACTGTCGATCTGGAGTATGGAAGAGGTGAGTGGAATTCCGAGTGTAGAGGTGAAATTCG\nTAGATATTCGGAGGAACACCAGTGGCGAAGGCGGCTCACTGGTCCATTACTGACGCTGAGGTGCGAAAGCGTGGGGAGCA\nAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAATGTTAGCCGTCGGGCAGTATACTGTTCGGTGGCGCAC\nGTAACGCATTAAACATTCCGCCTGGGGAGTACGGTCGCAAGATTAAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCG\nGTGGAGCATGTGGTTTAATTCGAAGCAACGCGCAGAACCTTACCAGCCCTTGACATGCCCGGCTACTTGCAGAGATGCAA\nGGTTCTTCGGGGACCGGGACACAGGTGCTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAAC\nGAGCGCAACCCTCGCCCTTAGTTGCCAGCATTCAGTTGGGCACTCTAAGGGGACTGCCGGTGATAAGCCGAGAGGAAGGT\nGGGGATGACGTCAAGTCCTCATGGCCCTTACGGGCTGGGCTACACACGTGCTACAATGGTGGTGACAGTGGGCAGCGAGC\nACGCGAGTGTGAGCTAATCTCCAAAAGCCATCTCAGTTCGGATTGCACTCTGCAACTCGAGTGCATGAAGTTGGAATCGC\nTAGTAATCGCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCATGGGAGTTGGT\nTTTACCCGAAGGTAGTGCGCTAACCGCAAGGAGGCAGCTAACCACGGTAGGGTCAGCGACTGGGGTGAAGTCGTAACAAG\nGTAGCCGTAGGGGAACCTGCGGCTGGATCACCTCC\n>Acidobacterium_capsulatum_ATCC51196 accession=NR_074106.1 source=NCBI_Nucleotide role=reference species=\"Acidobacterium capsulatum ATCC 51196\"\nAGAGTTTGATCCTGGCTCAGAATCAACGCTGGCGGCGTGCCTAACACATGCAAGTCGAACAAGAAAGGGACTTCGGTCCT\nGAGTACAGTGGCGCACGGGTGAGTAACACGTGACTAACCTACCCTCGAGTGGGGAATAACTTCGGGAAACCGAGGCTAAT\nACCGCATAATACCCACGGGTCAAAGGAGCAATTCGCTTGAGGAGGGGGTCGCGGCCGATTAGCTAGTTGGCGGGGTAATG\nGCCCACCAAGGCAGTGATCGGTATCCGGCCTGAGAGGGCGCACGGACACACTGGAACTGAAACACGGTCCAGACTCCTAC\nGGGAGGCAGCAGTGGGGAATTTTGCGCAATGGGGGAAACCCTGACGCAGCAACGCCGCGTGGAGGATGAAGTCTCTTGGG\nACGTAAACTCCTTTCGATCGGAACGATTATGACGGTACCGGAAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGG\nTAATACGAGGGGGGCGAGCGTTGTTCGGAATTATTGGGCGTAAAGGGTGCGTAGGCGGTTCGGTAAGTTTGATGTGAAAT\nCTTCGGGCTCAACTCGAAGTCTGCATCGAAAACTGCCGGGCTTGAGTGTGGGAGAGGTGAGTGGAATTTCCGGTGTAGCG\nGTGAAATGCGTAGATATCGGAAGGAACACCTGTGGCGAAAGCGGCTCACTGGACCACAACTGACGCTGATGCACGAAAGC\nTAGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCTAGCCCTAAACGATGATCGCTTGGTGTGGCGGGTACCCAATCCC\nGTCGTGCCGTAGCTAACGCGTTAAGCGATCCGCCTGGGGAGTACGGTCGCAAGGCTGAAACTCAAAGGAATTGACGGGGG\nCCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGACGCAACGCGAAGAACCTTACCTGGGCTCGAAATGTAGTGGACCGG\nGGTAGAAATATCCCTTCCCCGCAAGGGGCTGCTATATAGGTGCTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGG\nGTTAAGTCCCGCAACGAGCGCAACCCTTATTGCCAGTTGCTACCATTTAGTTGAGCACTCTGGTGAGACCGCCTCGGATA\nACGGGGAGGAAGGTGGGGATGACGTCAAGTCCTCATGGCCTTTATGTCCAGGGCTACACACGTGCTACAATGGCCGGTAC\nAAACCGCCGCAAACCCGCGAGGGGGAGCTAATCGGAAAAAGCCGGCCTCAGTTCGGATTGTAGTCTGCAACTCGACTACA\nTGAAGCTGGAATCGCTAGTAATCGCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCAC\nATCACGAAAGTGGGTCGTACTAGAAGCGGGTGAGCCAACCGTAAGGAGGCAGCCTTCCAAGGTGTGATTCATGATTGGGG\nTGAAGTCGTAACAAGGTAGCCGTAGGAGAACCTGCGGCTGGATCACCTCCTTT\n",
  "pilot_16s_query_reads.fasta": ">Soil_ASV_A source_accession=NR_102783.2 source=teaching_cache role=query\nAGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCCTACTACATGCAAGTCGAGCGGACAGATGGGAGCTTGCTC\nCCTGATGTTAGCGGCGGACGGGTGAGTAACACGTGGGTAACCTGCCTGTAAGACTGGGATACCTCCGGGAAACCGGGGCT\nAATACCGGATGGTTGTTTGAACCGCATGGTTCAAACATAAAAGGTGGCTTCGGCTACCACTTACAGATGGACCCGCGGCG\nCATTAGCTAGTTGGTGAGGTAACGGCTCACCAAGGCGACGATGCGTAGCCGACCTGAGAGGGTGATCGGGCACACTGGGA\nCTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTAGGGAATCTTCCGCAATGGACGAAAGTCTGACGGAGCAACGCC\nGCGTGAGTGATGAAGGTTTTCGGATCGTAAAGCTCTGTTGTTAGGGAAGAACAAGTGCCGTTCGAATAGGGGGGTACCTT\nGACGGTACCTAACCAGAAAGCCACGGCTAACTACGTGCCA\n>Soil_ASV_B source_accession=D14513.1 source=teaching_cache role=query\nAGAGTTTGATCCTGGCTCAGAACGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGATCGCCCCGCAANNNNAGCGGCA\nGACGGGTGAGTAACGCGTGGGAACGTACCCTTTACTACGGAATAACGCAGGGAAACTTGTGCTAATACCGTATGTGCCCT\nTTGGGGGAAAGATTTATCGGTAAAGGATCGGCCCGCGTTGGATTAGCTAGTTGGTGGGGTCAAGGCCTACCAAGGCGACG\nATCCATAGCTGGTCTGAGAGGATGATCAGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGCAGTGGG\nGAATATTGGACAATGGGCGCAAGCCTGATCCAGCCATGCCGCGTGAGTGATGAAGGCCCTAGGGTTGTAAATCTCTTTCA\nCCGGAGAAGATAATGACGGTATCCGGAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGGTAATACGAAGGGGGCT\nAGCGTTGTTCGGAATTACTGGGCGTCAAGCGCACGTAGGC\n",
  "pilot_16s_metadata.csv": "label,role,species_or_query,accession,source_database,source_url,retrieved_date,phylum,color,soil_context,note\nBacillus_subtilis_168,reference,Bacillus subtilis subsp. subtilis strain 168,NR_102783.2,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2,2026-05-25,Bacillota,#E69F00,common soil and rhizosphere model bacterium,\"Gram-positive, spore-forming soil bacterium; useful classroom decomposer reference.\"\nPseudomonas_fluorescens_CCM2115,reference,Pseudomonas fluorescens strain CCM 2115,NR_115715.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_115715.1,2026-05-25,Pseudomonadota,#56B4E9,rhizosphere-associated soil bacterium,Common plant-root associated reference; useful contrast to Gram-positive taxa.\nStreptomyces_coelicolor_rrnD,reference,Streptomyces coelicolor,Y00411.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/Y00411.1,2026-05-25,Actinomycetota,#009E73,filamentous soil actinomycete,Classic soil actinomycete; illustrates that soil microbes are not a single close group.\nRhizobium_leguminosarum_IAM12609,reference,Rhizobium leguminosarum type strain IAM 12609,D14513.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/D14513.1,2026-05-25,Pseudomonadota,#56B4E9,root nodule and nitrogen-cycling context,Plant-associated nitrogen-cycle reference; sequence contains a few ambiguous N bases from the original record.\nAcidobacterium_capsulatum_ATCC51196,reference,Acidobacterium capsulatum ATCC 51196,NR_074106.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_074106.1,2026-05-25,Acidobacteriota,#CC79A7,acidic soil and broad soil ecology reference,Soil-relevant reference from a major soil-associated phylum.\nSoil_ASV_A,query,unknown soil ASV A,NR_102783.2,Teaching cache derived from NCBI reference,https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2,2026-05-25,Teaching query,#222222,class teaching query from a rhizosphere-style sample,Synthetic classroom read derived from the Bacillus subtilis 16S marker window with a few substitutions.\nSoil_ASV_B,query,unknown soil ASV B,D14513.1,Teaching cache derived from NCBI reference,https://www.ncbi.nlm.nih.gov/nuccore/D14513.1,2026-05-25,Teaching query,#222222,class teaching query from a root-associated soil sample,Synthetic classroom read derived from the Rhizobium leguminosarum 16S marker window with a few substitutions.\n",
  "pilot_16s_cached_hits.csv": "query_label,rank,reference_label,reference_accession,reference_species,compared_bases,differences,fraction_different,percent_identity_teaching_window,source,interpretation\nSoil_ASV_A,1,Bacillus_subtilis_168,NR_102783.2,Bacillus subtilis subsp. subtilis strain 168,520,4,0.007692,99.231,precomputed class-safe teaching hit table,closest cached reference\nSoil_ASV_A,2,Streptomyces_coelicolor_rrnD,Y00411.1,Streptomyces coelicolor,483,96,0.198758,80.124,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,3,Rhizobium_leguminosarum_IAM12609,D14513.1,Rhizobium leguminosarum type strain IAM 12609,450,98,0.217778,78.222,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,4,Acidobacterium_capsulatum_ATCC51196,NR_074106.1,Acidobacterium capsulatum ATCC 51196,470,106,0.225532,77.447,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,5,Pseudomonas_fluorescens_CCM2115,NR_115715.1,Pseudomonas fluorescens strain CCM 2115,503,119,0.236581,76.342,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,1,Rhizobium_leguminosarum_IAM12609,D14513.1,Rhizobium leguminosarum type strain IAM 12609,514,3,0.005837,99.416,precomputed class-safe teaching hit table,closest cached reference\nSoil_ASV_B,2,Pseudomonas_fluorescens_CCM2115,NR_115715.1,Pseudomonas fluorescens strain CCM 2115,480,90,0.187500,81.250,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,3,Streptomyces_coelicolor_rrnD,Y00411.1,Streptomyces coelicolor,513,102,0.198830,80.117,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,4,Bacillus_subtilis_168,NR_102783.2,Bacillus subtilis subsp. subtilis strain 168,490,102,0.208163,79.184,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,5,Acidobacterium_capsulatum_ATCC51196,NR_074106.1,Acidobacterium capsulatum ATCC 51196,506,112,0.221344,77.866,precomputed class-safe teaching hit table,lower-ranked cached reference\n",
  "pilot_16s_cached_blast.xml": "<?xml version=\"1.0\"?>\n<BlastOutput>\n  <BlastOutput_program>blastn</BlastOutput_program>\n  <BlastOutput_version>cached-teaching-blast-1.0</BlastOutput_version>\n  <BlastOutput_reference>Class-safe teaching XML generated from precomputed aligned-window hits; not a live NCBI BLAST run.</BlastOutput_reference>\n  <BlastOutput_db>soil_16s_class_cache</BlastOutput_db>\n  <BlastOutput_query-ID>soil_16s_teaching_queries</BlastOutput_query-ID>\n  <BlastOutput_query-def>soil_16s_teaching_queries</BlastOutput_query-def>\n  <BlastOutput_param><Parameters><Parameters_matrix>identity</Parameters_matrix></Parameters></BlastOutput_param>\n  <BlastOutput_iterations>\n    <Iteration>\n      <Iteration_iter-num>1</Iteration_iter-num>\n      <Iteration_query-ID>Soil_ASV_A</Iteration_query-ID>\n      <Iteration_query-def>Soil_ASV_A</Iteration_query-def>\n      <Iteration_query-len>520</Iteration_query-len>\n      <Iteration_hits>\n        <Hit>\n          <Hit_num>1</Hit_num>\n          <Hit_id>Bacillus_subtilis_168</Hit_id>\n          <Hit_def>Bacillus subtilis subsp. subtilis strain 168</Hit_def>\n          <Hit_accession>NR_102783.2</Hit_accession>\n          <Hit_len>520</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>1028</Hsp_bit-score>\n              <Hsp_evalue>1e-120</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>520</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>520</Hsp_hit-to>\n              <Hsp_identity>516</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>520</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>2</Hit_num>\n          <Hit_id>Streptomyces_coelicolor_rrnD</Hit_id>\n          <Hit_def>Streptomyces coelicolor</Hit_def>\n          <Hit_accession>Y00411.1</Hit_accession>\n          <Hit_len>483</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>678</Hsp_bit-score>\n              <Hsp_evalue>1e-80</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>483</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>483</Hsp_hit-to>\n              <Hsp_identity>387</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>483</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>3</Hit_num>\n          <Hit_id>Rhizobium_leguminosarum_IAM12609</Hit_id>\n          <Hit_def>Rhizobium leguminosarum type strain IAM 12609</Hit_def>\n          <Hit_accession>D14513.1</Hit_accession>\n          <Hit_len>450</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>606</Hsp_bit-score>\n              <Hsp_evalue>1e-75</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>450</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>450</Hsp_hit-to>\n              <Hsp_identity>352</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>450</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>4</Hit_num>\n          <Hit_id>Acidobacterium_capsulatum_ATCC51196</Hit_id>\n          <Hit_def>Acidobacterium capsulatum ATCC 51196</Hit_def>\n          <Hit_accession>NR_074106.1</Hit_accession>\n          <Hit_len>470</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>622</Hsp_bit-score>\n              <Hsp_evalue>1e-70</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>470</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>470</Hsp_hit-to>\n              <Hsp_identity>364</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>470</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>5</Hit_num>\n          <Hit_id>Pseudomonas_fluorescens_CCM2115</Hit_id>\n          <Hit_def>Pseudomonas fluorescens strain CCM 2115</Hit_def>\n          <Hit_accession>NR_115715.1</Hit_accession>\n          <Hit_len>503</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>649</Hsp_bit-score>\n              <Hsp_evalue>1e-65</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>503</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>503</Hsp_hit-to>\n              <Hsp_identity>384</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>503</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n      </Iteration_hits>\n    </Iteration>\n    <Iteration>\n      <Iteration_iter-num>2</Iteration_iter-num>\n      <Iteration_query-ID>Soil_ASV_B</Iteration_query-ID>\n      <Iteration_query-def>Soil_ASV_B</Iteration_query-def>\n      <Iteration_query-len>514</Iteration_query-len>\n      <Iteration_hits>\n        <Hit>\n          <Hit_num>1</Hit_num>\n          <Hit_id>Rhizobium_leguminosarum_IAM12609</Hit_id>\n          <Hit_def>Rhizobium leguminosarum type strain IAM 12609</Hit_def>\n          <Hit_accession>D14513.1</Hit_accession>\n          <Hit_len>514</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>1019</Hsp_bit-score>\n              <Hsp_evalue>1e-120</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>514</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>514</Hsp_hit-to>\n              <Hsp_identity>511</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>514</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>2</Hit_num>\n          <Hit_id>Pseudomonas_fluorescens_CCM2115</Hit_id>\n          <Hit_def>Pseudomonas fluorescens strain CCM 2115</Hit_def>\n          <Hit_accession>NR_115715.1</Hit_accession>\n          <Hit_len>480</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>690</Hsp_bit-score>\n              <Hsp_evalue>1e-80</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>480</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>480</Hsp_hit-to>\n              <Hsp_identity>390</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>480</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>3</Hit_num>\n          <Hit_id>Streptomyces_coelicolor_rrnD</Hit_id>\n          <Hit_def>Streptomyces coelicolor</Hit_def>\n          <Hit_accession>Y00411.1</Hit_accession>\n          <Hit_len>513</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>720</Hsp_bit-score>\n              <Hsp_evalue>1e-75</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>513</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>513</Hsp_hit-to>\n              <Hsp_identity>411</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>513</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>4</Hit_num>\n          <Hit_id>Bacillus_subtilis_168</Hit_id>\n          <Hit_def>Bacillus subtilis subsp. subtilis strain 168</Hit_def>\n          <Hit_accession>NR_102783.2</Hit_accession>\n          <Hit_len>490</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>674</Hsp_bit-score>\n              <Hsp_evalue>1e-70</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>490</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>490</Hsp_hit-to>\n              <Hsp_identity>388</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>490</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>5</Hit_num>\n          <Hit_id>Acidobacterium_capsulatum_ATCC51196</Hit_id>\n          <Hit_def>Acidobacterium capsulatum ATCC 51196</Hit_def>\n          <Hit_accession>NR_074106.1</Hit_accession>\n          <Hit_len>506</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>676</Hsp_bit-score>\n              <Hsp_evalue>1e-65</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>506</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>506</Hsp_hit-to>\n              <Hsp_identity>394</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>506</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n      </Iteration_hits>\n    </Iteration>\n  </BlastOutput_iterations>\n</BlastOutput>\n",
  "pilot_16s_abundance_table.csv": "sample_id,Soil_ASV_A,Soil_ASV_B,note\nRhizosphere_A,128,34,Bacillus-like ASV is more abundant in this toy sample.\nCompost_B,76,93,Both ASVs are detectable in the compost-style toy sample.\nRoot_Nodule_C,21,156,Rhizobium-like ASV is more abundant in this toy sample.\n",
  "pilot_16s_manifest.json": "{\n  \"title\": \"Class-safe soil 16S phylogeny pilot cache\",\n  \"retrieved_date\": \"2026-05-25\",\n  \"reference_count\": 5,\n  \"query_count\": 2,\n  \"cached_hit_table\": \"pilot_16s_cached_hits.csv\",\n  \"cached_blast_xml\": \"pilot_16s_cached_blast.xml\",\n  \"default_mode\": \"Use these cached files; do not require live BLAST or Entrez during class.\",\n  \"references\": [\n    {\n      \"label\": \"Bacillus_subtilis_168\",\n      \"accession\": \"NR_102783.2\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2\"\n    },\n    {\n      \"label\": \"Pseudomonas_fluorescens_CCM2115\",\n      \"accession\": \"NR_115715.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_115715.1\"\n    },\n    {\n      \"label\": \"Streptomyces_coelicolor_rrnD\",\n      \"accession\": \"Y00411.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/Y00411.1\"\n    },\n    {\n      \"label\": \"Rhizobium_leguminosarum_IAM12609\",\n      \"accession\": \"D14513.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/D14513.1\"\n    },\n    {\n      \"label\": \"Acidobacterium_capsulatum_ATCC51196\",\n      \"accession\": \"NR_074106.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_074106.1\"\n    }\n  ],\n  \"teaching_queries\": [\n    {\n      \"label\": \"Soil_ASV_A\",\n      \"derived_from\": \"NR_102783.2\",\n      \"caution\": \"Synthetic classroom query; not a new environmental isolate.\"\n    },\n    {\n      \"label\": \"Soil_ASV_B\",\n      \"derived_from\": \"D14513.1\",\n      \"caution\": \"Synthetic classroom query; not a new environmental isolate.\"\n    }\n  ]\n}\n",
  "atacama_sample_metadata_mini.csv": "sample_id,transect_name,site_name,depth,elevation,average_soil_relative_humidity,vegetation,ph,toc,ec,percentcover\nBAQ1552.1.1,Baquedano,BAQ1552,1,1552,15.75,no,7.87,,,0\nBAQ2420.1.1,Baquedano,BAQ2420,1,2420,82.54,no,9.33,,,0\nBAQ2420.1.2,Baquedano,BAQ2420,2,2420,82.54,no,9.36,166,0.075,0\nBAQ2420.1.3,Baquedano,BAQ2420,3,2420,82.54,no,8.90,,,0\nBAQ2420.2,Baquedano,BAQ2420,2,2420,82.54,no,9.36,337,0.075,0\nBAQ2420.3,Baquedano,BAQ2420,2,2420,82.54,no,9.36,574,0.075,0\nBAQ2462.1,Baquedano,BAQ2462,2,2462,69.08,no,8.35,173,0.506,0\nBAQ2462.2,Baquedano,BAQ2462,2,2462,69.08,no,8.35,331,0.506,0\nBAQ2462.3,Baquedano,BAQ2462,2,2462,69.08,no,8.35,634,0.506,0\nBAQ2687.1,Baquedano,BAQ2687,2,2687,73.21,yes,8.35,554,0.131,0.1\nBAQ2687.2,Baquedano,BAQ2687,2,2687,73.21,yes,8.35,350,0.131,0.1\nBAQ2687.3,Baquedano,BAQ2687,2,2687,73.21,yes,8.35,400,0.131,0.1\nBAQ2838.1,Baquedano,BAQ2838,2,2838,44.74,no,8.36,353,0.212,0\nBAQ2838.2,Baquedano,BAQ2838,2,2838,44.74,no,8.36,260,0.212,0\nBAQ2838.3,Baquedano,BAQ2838,2,2838,44.74,no,8.36,369,0.212,0\nBAQ3473.1,Baquedano,BAQ3473,2,3473,82.05,yes,7.93,3675,0.42,1.2\nBAQ3473.2,Baquedano,BAQ3473,2,3473,82.05,yes,7.93,838,0.42,1.2\nBAQ3473.3,Baquedano,BAQ3473,2,3473,82.05,yes,7.93,16449,0.42,1.2\nBAQ4166.1.1,Baquedano,BAQ4166,1,4166,100,yes,5.64,,,7.1\nBAQ4166.1.2,Baquedano,BAQ4166,2,4166,100,yes,7.22,2085,0.084,7.1\nBAQ4166.1.3,Baquedano,BAQ4166,3,4166,100,yes,7.41,,,7.1\nBAQ4166.2,Baquedano,BAQ4166,2,4166,100,yes,7.22,3692,0.084,7.1\nBAQ4166.3,Baquedano,BAQ4166,2,4166,100,yes,7.22,2271,0.084,7.1\nBAQ4697.1,Baquedano,BAQ4697,2,4697,,yes,7.44,424,0.055,0.1\nBAQ4697.2,Baquedano,BAQ4697,2,4697,,yes,7.44,531,0.055,0.1\nBAQ4697.3,Baquedano,BAQ4697,2,4697,,yes,7.44,631,0.055,0.1\nYUN1005.1.1,Yungay,YUN1005,1,1005,20.7,no,7.54,,,0\nYUN1005.3,Yungay,YUN1005,2,1005,20.7,no,7.60,223,2.27,0\nYUN1242.1,Yungay,YUN1242,2,1242,20.9,no,9.00,226,1.845,0\nYUN1242.2,Yungay,YUN1242,2,1242,20.9,no,9.00,194,1.845,0\nYUN1242.3,Yungay,YUN1242,2,1242,20.9,no,9.00,452,1.845,0\nYUN1609.1,Yungay,YUN1609,2,1609,17.18,no,8.00,361,0.427,0\nYUN2029.1,Yungay,YUN2029,2,2029,28.79,no,7.89,184,0.067,0\nYUN2029.2,Yungay,YUN2029,2,2029,28.79,no,7.89,241,0.067,0\nYUN2029.3,Yungay,YUN2029,2,2029,28.79,no,7.89,817,0.067,0\nYUN3008.1.3,Yungay,YUN3008,3,3008,70.89,no,7.50,,,0\nYUN3008.3,Yungay,YUN3008,2,3008,70.89,no,,417,2.205,0\nYUN3153.2,Yungay,YUN3153,2,3153,59.69,no,7.60,185,2.26,0\nYUN3153.3,Yungay,YUN3153,2,3153,59.69,no,7.60,391,2.26,0\nYUN3184.2,Yungay,YUN3184,2,3184,26.97,no,,411,2.235,0\nYUN3259.1.1,Yungay,YUN3259,1,3259,93.57,yes,8.48,,,2.4\nYUN3259.1.2,Yungay,YUN3259,2,3259,93.57,yes,8.10,419,0.133,2.4\nYUN3259.1.3,Yungay,YUN3259,3,3259,93.57,yes,8.40,,,2.4\nYUN3259.2,Yungay,YUN3259,2,3259,93.57,yes,8.10,342,0.133,2.4\nYUN3259.3,Yungay,YUN3259,2,3259,93.57,yes,8.10,539,0.133,2.4\nYUN3346.1,Yungay,YUN3346,2,3346,87.32,yes,7.10,429,0.044,0.01\nYUN3346.2,Yungay,YUN3346,2,3346,87.32,yes,7.10,666,0.044,0.01\nYUN3346.3,Yungay,YUN3346,2,3346,87.32,yes,7.10,387,0.044,0.01\nYUN3428.1,Yungay,YUN3428,2,3428,99.99,yes,7.20,623,0.023,8.8\nYUN3428.2,Yungay,YUN3428,2,3428,99.99,yes,7.20,621,0.023,8.8\nYUN3428.3,Yungay,YUN3428,2,3428,99.99,yes,7.20,658,0.023,8.8\nYUN3533.1.1,Yungay,YUN3533,1,3533,100,yes,7.10,,,8.6\nYUN3533.1.2,Yungay,YUN3533,2,3533,100,yes,8.00,521,0.024,8.6\nYUN3533.1.3,Yungay,YUN3533,3,3533,100,yes,7.80,,,8.6\nYUN3533.2,Yungay,YUN3533,2,3533,100,yes,8.00,773,0.024,8.6\nYUN3533.3,Yungay,YUN3533,2,3533,100,yes,8.00,664,0.024,8.6\nYUN3856.1.1,Yungay,YUN3856,1,3856,99.44,yes,6.98,,,3.1\nYUN3856.1.2,Yungay,YUN3856,2,3856,99.44,yes,7.43,713,0.029,3.1\nYUN3856.1.3,Yungay,YUN3856,3,3856,99.44,yes,7.54,,,3.1\nYUN3856.2,Yungay,YUN3856,2,3856,99.44,yes,7.43,404,0.029,3.1\nYUN3856.3,Yungay,YUN3856,2,3856,99.44,yes,7.43,988,0.029,3.1\n",
  "atacama_feature_table_top12.csv": "sample_id,Atacama_ASV_01,Atacama_ASV_02,Atacama_ASV_03,Atacama_ASV_04,Atacama_ASV_05,Atacama_ASV_06,Atacama_ASV_07,Atacama_ASV_08,Atacama_ASV_09,Atacama_ASV_10,Atacama_ASV_11,Atacama_ASV_12,other_asvs,total_reads\nBAQ1552.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nBAQ2420.1.1,0,0,0,0,0,0,56,15,0,0,0,0,114,185\nBAQ2420.1.2,0,0,0,0,0,0,0,0,192,0,0,0,102,294\nBAQ2420.1.3,0,0,0,0,0,12,0,0,185,0,0,0,113,310\nBAQ2420.2,0,0,0,0,0,0,0,0,0,0,0,14,282,296\nBAQ2420.3,0,0,0,30,0,0,19,0,0,0,0,0,158,207\nBAQ2462.1,0,0,0,30,0,36,0,0,0,0,0,0,401,467\nBAQ2462.2,0,0,0,17,0,0,35,0,0,0,0,0,215,267\nBAQ2462.3,0,0,0,13,0,36,40,0,0,0,0,0,127,216\nBAQ2687.1,0,0,0,0,0,0,0,0,0,0,0,0,266,266\nBAQ2687.2,0,0,0,0,0,0,0,0,0,0,0,0,379,379\nBAQ2687.3,0,0,0,0,0,0,0,0,0,0,0,0,167,167\nBAQ2838.1,0,0,0,0,0,20,0,0,0,0,0,0,118,138\nBAQ2838.2,0,0,0,0,0,5,0,0,0,0,0,0,38,43\nBAQ2838.3,0,0,0,0,0,0,41,0,0,0,0,0,41,82\nBAQ3473.1,0,254,0,0,0,0,0,0,0,0,0,0,235,489\nBAQ3473.2,193,0,0,0,0,0,0,0,0,0,0,0,77,270\nBAQ3473.3,0,190,0,0,0,0,0,25,0,0,0,0,314,529\nBAQ4166.1.1,0,0,0,0,0,0,0,0,0,0,0,0,365,365\nBAQ4166.1.2,0,70,0,0,0,0,0,0,0,0,0,0,322,392\nBAQ4166.1.3,0,0,0,0,0,0,0,0,0,0,0,0,390,390\nBAQ4166.2,0,0,0,0,0,0,0,0,0,0,0,0,335,335\nBAQ4166.3,0,0,0,0,0,0,0,221,0,0,0,0,260,481\nBAQ4697.1,0,33,0,0,155,0,0,0,0,64,0,0,242,494\nBAQ4697.2,0,0,0,0,36,0,0,0,0,139,0,0,224,399\nBAQ4697.3,0,32,0,0,183,0,0,7,0,70,0,0,372,664\nYUN1005.1.1,0,0,0,0,0,0,0,0,0,0,0,0,578,578\nYUN1005.3,0,0,0,0,0,0,0,0,0,0,0,0,278,278\nYUN1242.1,0,0,0,0,0,0,0,0,0,0,0,0,376,376\nYUN1242.2,0,0,0,0,0,0,0,0,0,0,0,0,1,1\nYUN1242.3,0,0,0,0,0,53,36,0,0,0,0,0,400,489\nYUN1609.1,0,0,0,0,0,286,0,0,0,0,0,0,179,465\nYUN2029.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN2029.2,0,0,0,0,0,0,95,0,0,0,84,0,380,559\nYUN2029.3,0,0,0,0,0,0,0,0,0,0,0,0,1,1\nYUN3008.1.3,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3008.3,0,0,0,0,0,0,0,0,0,0,0,0,1,1\nYUN3153.2,0,0,0,0,0,0,0,0,0,0,0,0,190,190\nYUN3153.3,0,0,0,0,0,0,46,5,0,0,20,0,263,334\nYUN3184.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3259.1.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0\nYUN3259.1.2,0,0,0,0,0,0,0,0,0,0,0,36,324,360\nYUN3259.1.3,0,0,0,0,0,0,0,0,0,0,0,0,47,47\nYUN3259.2,0,0,0,0,0,0,0,20,0,0,0,0,162,182\nYUN3259.3,0,0,0,0,0,0,0,24,0,0,224,48,251,547\nYUN3346.1,0,0,0,0,0,0,0,0,0,0,0,114,291,405\nYUN3346.2,80,0,0,0,0,0,0,0,0,0,0,0,0,80\nYUN3346.3,0,0,0,0,0,0,0,0,0,0,0,40,333,373\nYUN3428.1,24,0,0,0,0,0,0,0,0,41,0,0,281,346\nYUN3428.2,0,0,0,0,0,0,0,0,0,0,0,50,416,466\nYUN3428.3,0,0,0,0,0,0,0,0,0,0,0,0,294,294\nYUN3533.1.1,0,0,319,0,0,0,0,0,0,0,0,0,172,491\nYUN3533.1.2,197,37,0,0,0,0,0,29,0,15,0,0,119,397\nYUN3533.1.3,187,0,0,27,0,0,0,0,0,0,0,0,173,387\nYUN3533.2,28,0,0,454,0,0,0,0,0,24,0,0,133,639\nYUN3533.3,47,0,0,0,0,0,0,24,0,23,0,0,340,434\nYUN3856.1.1,0,0,257,0,0,0,0,0,0,0,0,0,165,422\nYUN3856.1.2,258,0,0,0,0,0,0,0,0,0,0,0,242,500\nYUN3856.1.3,0,0,0,0,0,8,0,0,0,0,0,0,267,275\nYUN3856.2,203,0,0,0,0,0,0,27,0,0,0,0,270,500\nYUN3856.3,0,0,0,0,188,0,52,0,0,0,0,0,361,601\n",
  "atacama_relative_abundance_top12.csv": "sample_id,Atacama_ASV_01,Atacama_ASV_02,Atacama_ASV_03,Atacama_ASV_04,Atacama_ASV_05,Atacama_ASV_06,Atacama_ASV_07,Atacama_ASV_08,Atacama_ASV_09,Atacama_ASV_10,Atacama_ASV_11,Atacama_ASV_12,other_asvs,total_reads\nBAQ1552.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0\nBAQ2420.1.1,0.0,0.0,0.0,0.0,0.0,0.0,30.27027,8.10811,0.0,0.0,0.0,0.0,61.62162,185\nBAQ2420.1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,65.30612,0.0,0.0,0.0,34.69388,294\nBAQ2420.1.3,0.0,0.0,0.0,0.0,0.0,3.87097,0.0,0.0,59.67742,0.0,0.0,0.0,36.45161,310\nBAQ2420.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.72973,95.27027,296\nBAQ2420.3,0.0,0.0,0.0,14.49275,0.0,0.0,9.17874,0.0,0.0,0.0,0.0,0.0,76.3285,207\nBAQ2462.1,0.0,0.0,0.0,6.42398,0.0,7.70878,0.0,0.0,0.0,0.0,0.0,0.0,85.86724,467\nBAQ2462.2,0.0,0.0,0.0,6.36704,0.0,0.0,13.10861,0.0,0.0,0.0,0.0,0.0,80.52434,267\nBAQ2462.3,0.0,0.0,0.0,6.01852,0.0,16.66667,18.51852,0.0,0.0,0.0,0.0,0.0,58.7963,216\nBAQ2687.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,266\nBAQ2687.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,379\nBAQ2687.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,167\nBAQ2838.1,0.0,0.0,0.0,0.0,0.0,14.49275,0.0,0.0,0.0,0.0,0.0,0.0,85.50725,138\nBAQ2838.2,0.0,0.0,0.0,0.0,0.0,11.62791,0.0,0.0,0.0,0.0,0.0,0.0,88.37209,43\nBAQ2838.3,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,50.0,82\nBAQ3473.1,0.0,51.94274,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.05726,489\nBAQ3473.2,71.48148,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.51852,270\nBAQ3473.3,0.0,35.91682,0.0,0.0,0.0,0.0,0.0,4.7259,0.0,0.0,0.0,0.0,59.35728,529\nBAQ4166.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,365\nBAQ4166.1.2,0.0,17.85714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.14286,392\nBAQ4166.1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,390\nBAQ4166.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,335\nBAQ4166.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.94595,0.0,0.0,0.0,0.0,54.05405,481\nBAQ4697.1,0.0,6.68016,0.0,0.0,31.37652,0.0,0.0,0.0,0.0,12.95547,0.0,0.0,48.98785,494\nBAQ4697.2,0.0,0.0,0.0,0.0,9.02256,0.0,0.0,0.0,0.0,34.83709,0.0,0.0,56.14035,399\nBAQ4697.3,0.0,4.81928,0.0,0.0,27.56024,0.0,0.0,1.05422,0.0,10.54217,0.0,0.0,56.0241,664\nYUN1005.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,578\nYUN1005.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,278\nYUN1242.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,376\nYUN1242.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,1\nYUN1242.3,0.0,0.0,0.0,0.0,0.0,10.83845,7.36196,0.0,0.0,0.0,0.0,0.0,81.79959,489\nYUN1609.1,0.0,0.0,0.0,0.0,0.0,61.50538,0.0,0.0,0.0,0.0,0.0,0.0,38.49462,465\nYUN2029.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0\nYUN2029.2,0.0,0.0,0.0,0.0,0.0,0.0,16.99463,0.0,0.0,0.0,15.02683,0.0,67.97853,559\nYUN2029.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,1\nYUN3008.1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0\nYUN3008.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,1\nYUN3153.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,190\nYUN3153.3,0.0,0.0,0.0,0.0,0.0,0.0,13.77246,1.49701,0.0,0.0,5.98802,0.0,78.74251,334\nYUN3184.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0\nYUN3259.1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0\nYUN3259.1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,90.0,360\nYUN3259.1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,47\nYUN3259.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.98901,0.0,0.0,0.0,0.0,89.01099,182\nYUN3259.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.38757,0.0,0.0,40.95064,8.77514,45.88665,547\nYUN3346.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.14815,71.85185,405\nYUN3346.2,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,80\nYUN3346.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.72386,89.27614,373\nYUN3428.1,6.93642,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.84971,0.0,0.0,81.21387,346\nYUN3428.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.72961,89.27039,466\nYUN3428.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,294\nYUN3533.1.1,0.0,0.0,64.96945,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.03055,491\nYUN3533.1.2,49.62217,9.3199,0.0,0.0,0.0,0.0,0.0,7.30479,0.0,3.77834,0.0,0.0,29.97481,397\nYUN3533.1.3,48.32041,0.0,0.0,6.97674,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44.70284,387\nYUN3533.2,4.38185,0.0,0.0,71.04851,0.0,0.0,0.0,0.0,0.0,3.75587,0.0,0.0,20.81377,639\nYUN3533.3,10.82949,0.0,0.0,0.0,0.0,0.0,0.0,5.52995,0.0,5.29954,0.0,0.0,78.34101,434\nYUN3856.1.1,0.0,0.0,60.90047,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.09953,422\nYUN3856.1.2,51.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.4,500\nYUN3856.1.3,0.0,0.0,0.0,0.0,0.0,2.90909,0.0,0.0,0.0,0.0,0.0,0.0,97.09091,275\nYUN3856.2,40.6,0.0,0.0,0.0,0.0,0.0,0.0,5.4,0.0,0.0,0.0,0.0,54.0,500\nYUN3856.3,0.0,0.0,0.0,0.0,31.2812,0.0,8.65225,0.0,0.0,0.0,0.0,0.0,60.06656,601\n",
  "atacama_feature_key.csv": "asv_label,qiime_feature_id,total_reads,prevalence_samples,representative_sequence_length\nAtacama_ASV_01,409faa5f5353e543bf6d99125c7c0e83,1217,9,227\nAtacama_ASV_02,ffd60d684f32e6fd5b47fe90095f9d34,616,6,227\nAtacama_ASV_03,c434ee7f909f455fc2109ef1f741a2d4,576,2,227\nAtacama_ASV_04,1237d5925a7176fced9dda961a86c684,571,6,227\nAtacama_ASV_05,1694836fc379411d2b9aa087d68d571b,562,4,227\nAtacama_ASV_06,a7b877ae6d2f079a15b6b192a4425620,456,8,227\nAtacama_ASV_07,a36b38f754f6abd278aeb9dbc7696343,420,9,227\nAtacama_ASV_08,ef3fdbe1dcde754d91130cde6a4b4d61,397,10,227\nAtacama_ASV_09,9279403ab31c9b2574b09cef10f08586,377,2,227\nAtacama_ASV_10,96cbccca68ad868a78bb0604e4a41cf5,376,7,227\nAtacama_ASV_11,5987b97497dc2c6f7c3d690526ea34b0,328,3,227\nAtacama_ASV_12,11f7b172e09b77715d3bb2175a40b409,302,6,227\n",
  "atacama_top_asv_sequences.fasta": ">Atacama_ASV_01 qiime_feature_id=409faa5f5353e543bf6d99125c7c0e83\nAGCGTTAATCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGTTAGGTAAGTCGGATGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACTGTCTATCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_02 qiime_feature_id=ffd60d684f32e6fd5b47fe90095f9d34\nAGCGTTGTCCGGAATCATTGGGCGTAAAGAGCGTGTAGGCGGTCCGGTAAGTCGGCTGTGAAAGTCCAGGGCTCAACCCT\nGGGATGCCGGTCGATACTGCCGGACTAGAGTTCGGAAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCTATGGCGAAGGCAGCTCGCTGGGACGTTACTGACGCTGAGACGCGAAAGCGTG\n>Atacama_ASV_03 qiime_feature_id=c434ee7f909f455fc2109ef1f741a2d4\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTCTGTCGCGTCGGCTGTGAAAACTCGGGGCTCAACTCC\nGAGCTTGCAGTCGATACGGGCAGGCTAGAGTTCGGCAGGGGAGACTGGAATTCCTGGTGTAGCGGTGAAATGCGCAGATA\nTCAGGAGGAACACCGGTGGCGAAGGCGGGTCTCTGGGCCGATACTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_04 qiime_feature_id=1237d5925a7176fced9dda961a86c684\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTTGGGTAAGTCGGGTGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACTACCTAGCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_05 qiime_feature_id=1694836fc379411d2b9aa087d68d571b\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTAAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_06 qiime_feature_id=a7b877ae6d2f079a15b6b192a4425620\nAGCGTTGTCCGGATTTATTGGGCGTAAAGAGCTCGTAGGCGGCCTGGTGAGTCGGGTGTGAAAGCCCGAGGCTCAACCTC\nGGAATTGCATTCGATACTGCTGGGCTTGAGGCAGGTAGGGGAGGATGGAATTCCCGGTGTAGCGGTGGAATGCGCAGATA\nTCGGGAGGAACACCTGCGGCGAAGGCGGTCCTCTGGGCCTGTCCTGACGCTGAGGAGCGAAAGCGTG\n>Atacama_ASV_07 qiime_feature_id=a36b38f754f6abd278aeb9dbc7696343\nAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGTTCGGTAAGTCTGCCGTGAAAACCTGGGGCTCAACCCC\nGGGCGTGCGGTGGATACTGCCGGGCTAGAGGATGGTAGAGGCGAGTGGAATTCCCGGTGTAGCGGTGAAATGCGCAGATA\nTCGGGAGGAACACCAGTAGCGAAGGCGGCTCGCTGGGCCATTCCTGACGCTGAGACGCGAAAGCTAG\n>Atacama_ASV_08 qiime_feature_id=ef3fdbe1dcde754d91130cde6a4b4d61\nAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCTGATAAGTAGGGGGTGAAATCCTGCGGCTTAACCGC\nAGGGCTGCCTTCTAAACTGTCAGGCTCGAGCACAGTAGAGGCAGGTGGAATTCCCGGTGTAGCGGTGGAATGCGTAGAGA\nTCGGGAAGAACATCAGTGGCGAAGGCGGCCTGCTGGGCTGTTGCTGACGCTGAGGCGCGACAGCGTG\n>Atacama_ASV_09 qiime_feature_id=9279403ab31c9b2574b09cef10f08586\nAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGCTTACTAAGTCTGGTGTGAAAGCCCACGGCTCAACCGT\nGGAGGGCCATTGGAAACTGGTAGGCTTGAGTGCAGGAGAGGAGAGCGGAATTCCCGGTGTAGCGGTGAAATGCGTAGATA\nTCGGGAGGAACACCCGTGGCGAAGGCGGCTCTCTGGCCTGTAACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_10 qiime_feature_id=96cbccca68ad868a78bb0604e4a41cf5\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTAAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATTCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_11 qiime_feature_id=5987b97497dc2c6f7c3d690526ea34b0\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTTGGGTAAGTCGGGTGTGAAAGCCCTGGGCTTAACCTG\nGGAATGGCATTCGAGACCACCTATCTAGAGTCTGGTAGAGGGAAGTGGAATTTCCGGTGTAGCGGTGAAATGTGTAGATA\nTCGGAAGGAACACCAGTGGCGAAGGCGACTTCCTGGACCAAGACTGACGCTGAGGCGCGAAAGCGTG\n>Atacama_ASV_12 qiime_feature_id=11f7b172e09b77715d3bb2175a40b409\nAGCGTTAATCGGAATTACTGGGCGTAAAGGGCGCGTAGGCGGTGAAGTCAGTCGGGTGTGAAAGCCCCGGGCTCAACCTG\nGGAACTGCATCCGATACTGCTTCGCTAGAGTATGGTAGAGGGAAGCGGAATTCCGGGTGTAGCGGTGAAATGCGTAGATA\nTCCGGAGGAACACCAGTGGCGAAGGCGGCTTCCTGGACCAATACTGACGCTGAGGCGCGAAAGCGTG\n",
  "atacama_top_asv_stats.csv": "asv_label,mean_relative_abundance_percent,max_relative_abundance_percent,spearman_rho_vs_humidity,spearman_p_vs_humidity,spearman_q_bh_vs_humidity,mannwhitney_p_vegetated_vs_unvegetated,mannwhitney_q_bh_vegetated_vs_unvegetated\nAtacama_ASV_01,6.85307,100.0,0.37931,0.005092606071646548,0.03745139039487091,0.009758587320065766,0.039034349280263066\nAtacama_ASV_02,2.25957,51.94274,0.14534,0.2991042502098928,0.4595347106872555,0.040694900283476707,0.09766776068034409\nAtacama_ASV_03,2.24768,64.96945,0.21267,0.12629810117738463,0.25259620235476926,0.26194760498834324,0.2857610236236472\nAtacama_ASV_04,1.98799,71.04851,0.04065,0.7725718133281492,0.8428056145397992,0.1841903617702607,0.24558714902701428\nAtacama_ASV_05,1.77215,31.37652,0.10022,0.47522468901331016,0.5702696268159722,0.10218495855957313,0.17517421467355396\nAtacama_ASV_06,2.31464,61.50538,-0.35203,0.00973629253962709,0.03894517015850836,0.002213025679986303,0.013278154079917818\nAtacama_ASV_07,2.99745,50.0,-0.29452,0.03229490410048905,0.09688471230146714,0.0008224734748150264,0.009869681697780316\nAtacama_ASV_08,1.6954,45.94595,0.24336,0.07909988778533263,0.1898397306847983,0.17089033736030024,0.24558714902701428\nAtacama_ASV_09,2.23185,65.30612,-0.01301,0.9263449391876171,0.9263449391876172,0.0804777953788197,0.1609555907576394\nAtacama_ASV_10,1.48247,34.83709,0.37099,0.006241898399145151,0.03745139039487091,0.02550702354407182,0.07652107063221546\nAtacama_ASV_11,1.10653,40.95064,-0.14319,0.30635647379150366,0.4595347106872555,0.3553384810788416,0.3553384810788416\nAtacama_ASV_12,1.30547,28.14815,0.1319,0.3464722356259249,0.4619629808345665,0.2113552825207653,0.2536263390249184\n",
  "atacama_alpha_diversity.csv": "sample_id,total_reads,observed_asvs,shannon_entropy\nBAQ1552.1.1,0,0,0.0\nBAQ2420.1.1,185,8,1.78\nBAQ2420.1.2,294,8,1.17438\nBAQ2420.1.3,310,10,1.48871\nBAQ2420.2,296,14,2.23713\nBAQ2420.3,207,11,2.06362\nBAQ2462.1,467,12,2.06846\nBAQ2462.2,267,12,2.27134\nBAQ2462.3,216,11,2.10769\nBAQ2687.1,266,9,2.08871\nBAQ2687.2,379,10,2.04318\nBAQ2687.3,167,10,2.18702\nBAQ2838.1,138,10,2.17046\nBAQ2838.2,43,4,1.20709\nBAQ2838.3,82,6,1.34959\nBAQ3473.1,489,11,1.66974\nBAQ3473.2,270,7,1.09648\nBAQ3473.3,529,16,2.27115\nBAQ4166.1.1,365,10,1.70418\nBAQ4166.1.2,392,10,2.05642\nBAQ4166.1.3,390,14,2.11929\nBAQ4166.2,335,13,2.44902\nBAQ4166.3,481,13,1.91312\nBAQ4697.1,494,14,2.26676\nBAQ4697.2,399,13,2.16252\nBAQ4697.3,664,20,2.53605\nYUN1005.1.1,578,17,2.17925\nYUN1005.3,278,8,1.72587\nYUN1242.1,376,8,1.72258\nYUN1242.2,1,1,-0.0\nYUN1242.3,489,14,2.23989\nYUN1609.1,465,8,1.34966\nYUN2029.1,0,0,0.0\nYUN2029.2,559,16,2.3881\nYUN2029.3,1,1,-0.0\nYUN3008.1.3,0,0,0.0\nYUN3008.3,1,1,-0.0\nYUN3153.2,190,7,1.67644\nYUN3153.3,334,13,2.3064\nYUN3184.2,0,0,0.0\nYUN3259.1.1,0,0,0.0\nYUN3259.1.2,360,9,1.84188\nYUN3259.1.3,47,2,0.6655\nYUN3259.2,182,9,1.75341\nYUN3259.3,547,13,1.99206\nYUN3346.1,405,15,2.30531\nYUN3346.2,80,1,-0.0\nYUN3346.3,373,10,1.71954\nYUN3428.1,346,13,2.39161\nYUN3428.2,466,17,2.63405\nYUN3428.3,294,9,1.42648\nYUN3533.1.1,491,9,1.29614\nYUN3533.1.2,397,12,1.82639\nYUN3533.1.3,387,11,1.7663\nYUN3533.2,639,12,1.24312\nYUN3533.3,434,19,2.72109\nYUN3856.1.1,422,9,1.40613\nYUN3856.1.2,500,15,1.91681\nYUN3856.1.3,275,14,2.10657\nYUN3856.2,500,18,2.27841\nYUN3856.3,601,25,2.66241\n",
  "atacama_alpha_diversity_stats.csv": "metric,spearman_rho_vs_humidity,spearman_p_vs_humidity,spearman_q_bh_vs_humidity\ntotal_reads,0.32066,0.019233999270429764,0.028850998905644645\nobserved_asvs,0.33714,0.013562285245834537,0.028850998905644645\nshannon_entropy,0.13724,0.32711268859115705,0.32711268859115705\n",
  "atacama_mini_manifest.json": "{\n  \"title\": \"Atacama soil microbiome mini-cache for browser-only teaching\",\n  \"source_tutorial\": \"QIIME 2 2024.10 Atacama soil microbiome tutorial and q2-vsearch chimera tutorial\",\n  \"source_study\": \"Neilson et al. 2017 mSystems, Significant Impacts of Increasing Aridity on the Arid Soil Microbiome\",\n  \"source_doi\": \"10.1128/mSystems.00195-16\",\n  \"source_urls\": {\n    \"sample_metadata\": \"https://data.qiime2.org/2024.10/tutorials/atacama-soils/sample_metadata.tsv\",\n    \"feature_table\": \"https://data.qiime2.org/2024.10/tutorials/chimera/atacama-table.qza\",\n    \"representative_sequences\": \"https://data.qiime2.org/2024.10/tutorials/chimera/atacama-rep-seqs.qza\"\n  },\n  \"sample_count\": 61,\n  \"feature_count_full_table\": 401,\n  \"top_feature_count_in_cache\": 12,\n  \"teaching_use\": \"Use for real soil ASV abundance, humidity/vegetation association, and adjusted p-value teaching without installing QIIME 2.\"\n}\n"
}

OKABE_ITO = {
    "orange": "#E69F00",
    "sky_blue": "#56B4E9",
    "bluish_green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "reddish_purple": "#CC79A7",
    "black": "#222222",
    "gray": "#DDDDDD",
}

BASE_COLORS = {
    "A": OKABE_ITO["sky_blue"],
    "C": OKABE_ITO["bluish_green"],
    "G": OKABE_ITO["orange"],
    "T": OKABE_ITO["reddish_purple"],
    "N": "#BDBDBD",
    "-": "#E6E6E6",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#222222",
    "text.color": "#222222",
    "xtick.color": "#222222",
    "ytick.color": "#222222",
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "regular",
    "savefig.facecolor": "white",
})
sns.set_theme(style="white")

print("Ready. Dependencies are available and the embedded class cache is loaded in memory.")


## 1. Why cached data?

What can fail during a live class? A public database can be slow, a web API can throttle, or a student can lose time debugging a connection.

So this pilot uses a pre-cached reference set. The biology is still real enough for the lesson: the five reference records are NCBI 16S rRNA sequences from soil-relevant bacteria. The two query reads are clearly marked teaching reads derived from that cache.

**Think:** Why is cached data better for the first teaching run than live BLAST?


In [ ]:
#@title Load cache from GitHub or embedded fallback
def load_cache_file(filename):
    if USE_GITHUB_CACHE and CACHE_BASE_URL.strip():
        url = CACHE_BASE_URL.rstrip("/") + "/" + filename
        try:
            with urllib.request.urlopen(url, timeout=20) as handle:
                text = handle.read().decode("utf-8")
            print(f"Loaded {filename} from GitHub cache.")
            return text
        except Exception as exc:
            print(f"GitHub cache failed for {filename}: {exc}")
            print("Using embedded fallback copy.")
    return EMBEDDED_CACHE[filename]

ref_fasta = load_cache_file("pilot_16s_references.fasta")
query_fasta = load_cache_file("pilot_16s_query_reads.fasta")
metadata_csv = load_cache_file("pilot_16s_metadata.csv")
cached_hits_csv = load_cache_file("pilot_16s_cached_hits.csv")
cached_blast_xml = load_cache_file("pilot_16s_cached_blast.xml")
abundance_csv = load_cache_file("pilot_16s_abundance_table.csv")
manifest_json = load_cache_file("pilot_16s_manifest.json")
atacama_metadata_csv = load_cache_file("atacama_sample_metadata_mini.csv")
atacama_counts_csv = load_cache_file("atacama_feature_table_top12.csv")
atacama_relative_csv = load_cache_file("atacama_relative_abundance_top12.csv")
atacama_feature_key_csv = load_cache_file("atacama_feature_key.csv")
atacama_stats_csv = load_cache_file("atacama_top_asv_stats.csv")
atacama_alpha_csv = load_cache_file("atacama_alpha_diversity.csv")
atacama_alpha_stats_csv = load_cache_file("atacama_alpha_diversity_stats.csv")
atacama_manifest_json = load_cache_file("atacama_mini_manifest.json")

references = list(SeqIO.parse(StringIO(ref_fasta), "fasta"))
queries = list(SeqIO.parse(StringIO(query_fasta), "fasta"))
all_records = references + queries
metadata = pd.read_csv(StringIO(metadata_csv))
cached_hits = pd.read_csv(StringIO(cached_hits_csv))
abundance = pd.read_csv(StringIO(abundance_csv))
manifest = json.loads(manifest_json)
atacama_metadata = pd.read_csv(StringIO(atacama_metadata_csv))
atacama_counts = pd.read_csv(StringIO(atacama_counts_csv))
atacama_relative = pd.read_csv(StringIO(atacama_relative_csv))
atacama_feature_key = pd.read_csv(StringIO(atacama_feature_key_csv))
atacama_stats = pd.read_csv(StringIO(atacama_stats_csv))
atacama_alpha = pd.read_csv(StringIO(atacama_alpha_csv))
atacama_alpha_stats = pd.read_csv(StringIO(atacama_alpha_stats_csv))
atacama_manifest = json.loads(atacama_manifest_json)

def parse_cached_blast_xml(xml_text):
    root = ET.fromstring(xml_text)
    rows = []
    for iteration in root.findall(".//Iteration"):
        query = iteration.findtext("Iteration_query-def")
        for hit in iteration.findall("./Iteration_hits/Hit"):
            hsp = hit.find("./Hit_hsps/Hsp")
            align_len = int(hsp.findtext("Hsp_align-len"))
            identity = int(hsp.findtext("Hsp_identity"))
            rows.append({
                "query_label": query,
                "rank": int(hit.findtext("Hit_num")),
                "reference_label": hit.findtext("Hit_id"),
                "reference_accession": hit.findtext("Hit_accession"),
                "reference_species": hit.findtext("Hit_def"),
                "compared_bases": align_len,
                "differences": align_len - identity,
                "percent_identity_teaching_window": 100 * identity / align_len,
                "bit_score": float(hsp.findtext("Hsp_bit-score")),
                "teaching_e_value": hsp.findtext("Hsp_evalue"),
            })
    return pd.DataFrame(rows)

cached_blast_hits = parse_cached_blast_xml(cached_blast_xml)

print(f"Reference sequences: {len(references)}")
print(f"Query reads: {len(queries)}")
print(f"Cached BLAST-like XML hits: {len(cached_blast_hits)}")
print(f"Atacama samples: {len(atacama_metadata)}")
print(f"Atacama top ASVs in cache: {len(atacama_stats)}")
print("Cache retrieved date:", manifest["retrieved_date"])


In [ ]:
#@title Show the teaching reference set
def wrapped_table(df, columns):
    rows = []
    for _, row in df[columns].iterrows():
        cells = "".join(
            f"<td style='padding:8px 10px; border-top:1px solid #e5e5e5; vertical-align:top;'>{row[col]}</td>"
            for col in columns
        )
        rows.append(f"<tr>{cells}</tr>")
    header = "".join(
        f"<th style='text-align:left; padding:7px 10px; border-bottom:1px solid #777;'>{col}</th>"
        for col in columns
    )
    html = f'''
    <div style='font-family: system-ui, Segoe UI, sans-serif; max-width: 980px;'>
      <table style='border-collapse: collapse; table-layout: fixed; width: 100%; font-size: 13px; line-height: 1.35;'>
        <thead><tr>{header}</tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    '''
    display(HTML(html))

display_cols = ["label", "role", "species_or_query", "phylum", "soil_context", "note"]
wrapped_table(metadata, display_cols)


## 2. The class-safe workflow

What should happen when a student presses **Run all**?

The notebook should load known files, make the same comparison for everyone, and produce interpretable figures. Live BLAST and live database searches are useful later, but they are not allowed to be the first-class path because they can fail during class.

The workflow is:

**cached references + cached query reads -> marker window -> alignment view -> distance matrix -> UPGMA/NJ trees -> cautious report**


In [ ]:
#@title Workflow map
steps = [
    ("cache", "cached FASTA + metadata"),
    ("window", "shared 16S marker window"),
    ("align", "aligned position view"),
    ("distance", "pairwise distances"),
    ("tree", "UPGMA + NJ trees"),
    ("claim", "closest-reference report"),
]

fig, ax = plt.subplots(figsize=(11, 2.2))
ax.set_xlim(-0.5, len(steps) - 0.5)
ax.set_ylim(-0.5, 1.05)
ax.axis("off")

for i, (short, label) in enumerate(steps):
    ax.scatter(i, 0.35, s=160, color=OKABE_ITO["sky_blue"], edgecolor="#222222", linewidth=0.8, zorder=3)
    ax.text(i, 0.72, label, ha="center", va="bottom", fontsize=9, wrap=True)
    ax.text(i, 0.35, str(i + 1), ha="center", va="center", fontsize=9, color="white", weight="bold")
    if i < len(steps) - 1:
        ax.plot([i + 0.15, i + 0.85], [0.35, 0.35], color="#999999", linewidth=1.0, zorder=1)

ax.set_title("Class-safe path: no live web service is required during the lesson", loc="left", fontsize=12)
plt.tight_layout()
plt.show()


## 3. Where the references come from

In a real soil microbiome project, a 16S read can be compared against databases such as NCBI BLAST/GenBank, SILVA, RDP, and GTDB.

For this pilot, we already did the retrieval step before class and cached a small reference set. That makes the first run reliable. Later, the class can replace the teaching cache with references selected from live searches.


In [ ]:
#@title Reference database map
database_rows = pd.DataFrame([
    {
        "resource": "NCBI BLAST / GenBank",
        "helps with": "finding close public sequence records and accessions",
        "class-safe use": "cache selected FASTA and source links before class",
    },
    {
        "resource": "SILVA",
        "helps with": "curated rRNA reference alignment and taxonomy context",
        "class-safe use": "download or export selected references before class",
    },
    {
        "resource": "RDP",
        "helps with": "ribosomal RNA taxonomy and classifier-style teaching comparisons",
        "class-safe use": "cache selected reference records or classifier output",
    },
    {
        "resource": "GTDB",
        "helps with": "modern genome-based bacterial taxonomy context",
        "class-safe use": "use as taxonomy background, not as a live dependency",
    },
])
wrapped_table(database_rows, ["resource", "helps with", "class-safe use"])


## 4. Cached closest-hit table

What would students normally wait for from BLAST?

They would wait for a ranked hit table. For class, we cache that idea too. This table is computed from the same teaching marker window and lets students see the species-finding result before they build a tree.

The E-values and bit scores shown below come from the cached BLAST-like teaching XML. They are included so students can learn how ranked-hit evidence is read, but they are not live NCBI BLAST statistics.


In [ ]:
#@title Cached closest-hit table
xml_top = (
    cached_blast_hits[cached_blast_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
csv_top = (
    cached_hits[cached_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
assert xml_top == csv_top, f"Cached XML and CSV disagree: {xml_top} vs {csv_top}"

hit_view = (
    cached_blast_hits[cached_blast_hits["rank"] <= 3]
    .merge(
        cached_hits[["query_label", "rank", "fraction_different", "interpretation"]],
        on=["query_label", "rank"],
        how="left",
    )
    .copy()
)
hit_view["percent_identity_teaching_window"] = hit_view["percent_identity_teaching_window"].map(lambda x: f"{float(x):.2f}")
hit_view["fraction_different"] = hit_view["fraction_different"].map(lambda x: f"{float(x):.4f}")
hit_view["bit_score"] = hit_view["bit_score"].map(lambda x: f"{float(x):.1f}")
wrapped_table(
    hit_view[
        [
            "query_label",
            "rank",
            "reference_label",
            "reference_accession",
            "percent_identity_teaching_window",
            "bit_score",
            "teaching_e_value",
            "interpretation",
        ]
    ],
    [
        "query_label",
        "rank",
        "reference_label",
        "reference_accession",
        "percent_identity_teaching_window",
        "bit_score",
        "teaching_e_value",
        "interpretation",
    ],
)
print("Cached BLAST-like XML agrees with the cached hit CSV for the top hit of each query.")
print("E-values and bit scores here are cached teaching values, not fresh live-BLAST statistics.")


## 5. From soil sample to species-finding question

A microbiome table can tell us that a read or ASV exists in a sample. It does not automatically tell us what species it is.

The next question is narrower: which cached reference sequence is each query most similar to?

We answer that with sequence comparison first, then we use a tree to report relatedness.


In [ ]:
#@title Tiny microbiome-style count table
fig, ax = plt.subplots(figsize=(8.5, 3.2))
counts = abundance.set_index("sample_id")[["Soil_ASV_A", "Soil_ASV_B"]]
counts.plot(
    kind="bar",
    stacked=True,
    color=[OKABE_ITO["blue"], OKABE_ITO["orange"]],
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)
ax.set_ylabel("teaching read count")
ax.set_xlabel("")
ax.set_title("Toy soil samples: two ASVs to identify", loc="left", fontsize=12)
ax.legend(frameon=False, title="")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#eeeeee", linewidth=0.8)
plt.xticks(rotation=0, ha="center")
plt.tight_layout()
plt.show()
wrapped_table(abundance, ["sample_id", "Soil_ASV_A", "Soil_ASV_B", "note"])


## 6. Real soil ASV abundance: Atacama mini-cache

The toy table above is useful for the controlled species-finding exercise. To show a real soil microbiome pattern, this notebook also carries a small derived cache from the QIIME 2 Atacama soil microbiome tutorial.

Here the question changes: not "which species is this read closest to?" but "which ASVs vary with sample metadata such as soil humidity or vegetation?"

The q-values below are Benjamini-Hochberg adjusted p-values for abundance/metadata association tests. They are not tree branch support values.


In [ ]:
#@title Show the real-soil cache provenance
atacama_source_summary = pd.DataFrame([
    {
        "item": "source tutorial",
        "value": atacama_manifest["source_tutorial"],
    },
    {
        "item": "source study",
        "value": atacama_manifest["source_study"],
    },
    {
        "item": "DOI",
        "value": atacama_manifest["source_doi"],
    },
    {
        "item": "samples in mini-cache",
        "value": atacama_manifest["sample_count"],
    },
    {
        "item": "full table ASVs",
        "value": atacama_manifest["feature_count_full_table"],
    },
    {
        "item": "top ASVs used here",
        "value": atacama_manifest["top_feature_count_in_cache"],
    },
])
wrapped_table(atacama_source_summary, ["item", "value"])

atacama_stats_view = (
    atacama_stats
    .sort_values("spearman_q_bh_vs_humidity")
    [[
        "asv_label",
        "mean_relative_abundance_percent",
        "max_relative_abundance_percent",
        "spearman_rho_vs_humidity",
        "spearman_p_vs_humidity",
        "spearman_q_bh_vs_humidity",
        "mannwhitney_q_bh_vegetated_vs_unvegetated",
    ]]
    .head(8)
    .round(5)
)
display(atacama_stats_view)


In [ ]:
#@title Plot one Atacama ASV against soil humidity
selected_asv = ATACAMA_ASV_TO_PLOT
if selected_asv not in atacama_relative.columns:
    selected_asv = "Atacama_ASV_01"

atacama_plot = atacama_metadata.merge(atacama_relative, on="sample_id", how="inner")
atacama_plot["humidity"] = pd.to_numeric(
    atacama_plot["average_soil_relative_humidity"],
    errors="coerce",
)
atacama_plot["relative_abundance_percent"] = pd.to_numeric(
    atacama_plot[selected_asv],
    errors="coerce",
)
atacama_plot = atacama_plot.dropna(subset=["humidity", "relative_abundance_percent"])

stat_row = atacama_stats.loc[atacama_stats["asv_label"] == selected_asv].iloc[0]
veg_palette = {"yes": OKABE_ITO["bluish_green"], "no": OKABE_ITO["vermillion"]}
veg_labels = {"yes": "vegetated", "no": "unvegetated"}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.2),
    gridspec_kw={"width_ratios": [1.25, 1.0]},
)

ax = axes[0]
for vegetation_value, group in atacama_plot.groupby("vegetation"):
    ax.scatter(
        group["humidity"],
        group["relative_abundance_percent"],
        s=42,
        alpha=0.82,
        color=veg_palette.get(vegetation_value, OKABE_ITO["gray"]),
        edgecolor="white",
        linewidth=0.5,
        label=veg_labels.get(vegetation_value, vegetation_value),
    )
ax.set_xlabel("average soil relative humidity (%)")
ax.set_ylabel(f"{selected_asv} relative abundance (%)")
ax.set_title(
    f"{selected_asv}: abundance vs humidity",
    loc="left",
    fontsize=12,
)
ax.text(
    0.02,
    0.98,
    f"Spearman rho={stat_row['spearman_rho_vs_humidity']:.2f}; q={stat_row['spearman_q_bh_vs_humidity']:.3g}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 2},
)
ax.legend(frameon=False, title="")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#eeeeee", linewidth=0.8)

ax = axes[1]
q_rank = atacama_stats.sort_values("spearman_q_bh_vs_humidity").head(8).copy()
q_rank = q_rank.sort_values("spearman_q_bh_vs_humidity", ascending=True)
colors = [
    OKABE_ITO["blue"] if q <= 0.05 else "#9E9E9E"
    for q in q_rank["spearman_q_bh_vs_humidity"]
]
ax.barh(
    q_rank["asv_label"],
    q_rank["spearman_q_bh_vs_humidity"],
    color=colors,
    edgecolor="white",
    linewidth=0.7,
)
ax.axvline(0.05, color="#555555", linewidth=1.0)
ax.text(0.052, -0.45, "q=0.05", fontsize=9, color="#555555")
ax.set_xscale("log")
ax.invert_yaxis()
ax.set_xlabel("BH adjusted p-value for humidity association")
ax.set_ylabel("")
ax.set_title("Top Atacama ASV humidity tests", loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#eeeeee", linewidth=0.8)

plt.tight_layout()
plt.show()


In [ ]:
#@title Plot alpha diversity against humidity
alpha_plot = atacama_metadata.merge(atacama_alpha, on="sample_id", how="inner")
alpha_plot["humidity"] = pd.to_numeric(
    alpha_plot["average_soil_relative_humidity"],
    errors="coerce",
)
alpha_plot["observed_asvs"] = pd.to_numeric(alpha_plot["observed_asvs"], errors="coerce")
alpha_plot = alpha_plot.dropna(subset=["humidity", "observed_asvs"])

observed_row = (
    atacama_alpha_stats
    .loc[atacama_alpha_stats["metric"] == "observed_asvs"]
    .iloc[0]
)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for vegetation_value, group in alpha_plot.groupby("vegetation"):
    ax.scatter(
        group["humidity"],
        group["observed_asvs"],
        s=42,
        alpha=0.82,
        color=veg_palette.get(vegetation_value, OKABE_ITO["gray"]),
        edgecolor="white",
        linewidth=0.5,
        label=veg_labels.get(vegetation_value, vegetation_value),
    )
ax.set_xlabel("average soil relative humidity (%)")
ax.set_ylabel("observed ASVs")
ax.set_title("Atacama alpha diversity: observed ASVs vs humidity", loc="left", fontsize=12)
ax.text(
    0.02,
    0.98,
    f"Spearman rho={observed_row['spearman_rho_vs_humidity']:.2f}; q={observed_row['spearman_q_bh_vs_humidity']:.3g}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 2},
)
ax.legend(frameon=False, title="")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#eeeeee", linewidth=0.8)
plt.tight_layout()
plt.show()

display(atacama_alpha_stats.round(5))


## 7. Make a comparable 16S marker window

How can we compare sequences fairly? First, we choose the same marker region.

This pilot trims each reference to a shared 16S starting anchor and then uses Biopython's `PairwiseAligner` to make a small star alignment around one reference coordinate. That keeps the class path simple, but still makes the distance step depend on aligned columns rather than raw string positions.

In the later project notebook, this shortcut can be replaced by MAFFT on the team's cleaned reads and selected references.


In [ ]:
#@title Extract the class marker window
def clean_sequence(record):
    return str(record.seq).upper().replace("U", "T").replace(" ", "")

def marker_window(seq, max_bases):
    anchor = "AGAGTTTGATCCTGGCTCAG"
    start = seq.find(anchor)
    if start < 0:
        backup = seq.find("GAGTTTGATCCTGGCTCAG")
        start = max(backup - 1, 0) if backup >= 0 else 0
    return seq[start : start + max_bases]

windows = {}
for record in all_records:
    seq = clean_sequence(record)
    if record.id.startswith("Soil_ASV"):
        windows[record.id] = seq[:MARKER_WINDOW_BASES]
    else:
        windows[record.id] = marker_window(seq, MARKER_WINDOW_BASES)

def global_align_pair(reference, sequence):
    aligner = PairwiseAligner(
        mode="global",
        match_score=2,
        mismatch_score=-1,
        open_gap_score=-5,
        extend_gap_score=-0.5,
    )
    alignment = aligner.align(reference, sequence)[0]
    ref_blocks, seq_blocks = alignment.aligned
    ref_parts = []
    seq_parts = []
    ref_pos = 0
    seq_pos = 0

    for (ref_start, ref_end), (seq_start, seq_end) in zip(ref_blocks, seq_blocks):
        if ref_start > ref_pos:
            ref_parts.append(reference[ref_pos:ref_start])
            seq_parts.append("-" * (ref_start - ref_pos))
        if seq_start > seq_pos:
            ref_parts.append("-" * (seq_start - seq_pos))
            seq_parts.append(sequence[seq_pos:seq_start])

        ref_parts.append(reference[ref_start:ref_end])
        seq_parts.append(sequence[seq_start:seq_end])
        ref_pos = int(ref_end)
        seq_pos = int(seq_end)

    if ref_pos < len(reference):
        ref_parts.append(reference[ref_pos:])
        seq_parts.append("-" * (len(reference) - ref_pos))
    if seq_pos < len(sequence):
        ref_parts.append("-" * (len(sequence) - seq_pos))
        seq_parts.append(sequence[seq_pos:])

    aligned_ref = "".join(ref_parts)
    aligned_seq = "".join(seq_parts)
    if len(aligned_ref) != len(aligned_seq):
        raise ValueError("PairwiseAligner returned unequal reconstructed alignment lengths")
    return aligned_ref, aligned_seq


def star_align_to_reference(windows, reference_label):
    reference = windows[reference_label]
    ref_len = len(reference)
    base_by_label = {}
    insertions_by_label = {}
    max_insertions = {i: 0 for i in range(ref_len + 1)}

    for label, sequence in windows.items():
        if label == reference_label:
            aligned_ref, aligned_seq = reference, sequence
        else:
            aligned_ref, aligned_seq = global_align_pair(reference, sequence)

        bases = ["-"] * ref_len
        insertions = {}
        ref_pos = 0
        for ref_base, seq_base in zip(aligned_ref, aligned_seq):
            if ref_base == "-":
                insertions.setdefault(ref_pos, []).append(seq_base)
            else:
                if ref_pos < ref_len:
                    bases[ref_pos] = seq_base
                ref_pos += 1

        compact_insertions = {pos: "".join(chars) for pos, chars in insertions.items()}
        for pos, inserted in compact_insertions.items():
            max_insertions[pos] = max(max_insertions.get(pos, 0), len(inserted))
        base_by_label[label] = bases
        insertions_by_label[label] = compact_insertions

    aligned = {}
    for label in windows:
        chars = []
        for pos in range(ref_len):
            inserted = insertions_by_label[label].get(pos, "")
            chars.append(inserted.ljust(max_insertions.get(pos, 0), "-"))
            chars.append(base_by_label[label][pos])
        terminal_insert = insertions_by_label[label].get(ref_len, "")
        chars.append(terminal_insert.ljust(max_insertions.get(ref_len, 0), "-"))
        aligned[label] = "".join(chars)
    return aligned

aligned_windows = star_align_to_reference(windows, "Bacillus_subtilis_168")

window_lengths = pd.DataFrame({
    "raw_marker_bases": {label: len(seq) for label, seq in windows.items()},
    "aligned_columns": {label: len(seq) for label, seq in aligned_windows.items()},
})
display(window_lengths)
if window_lengths["raw_marker_bases"].min() < MARKER_WINDOW_BASES:
    print("Some records are shorter than the selected window; distances use the available bases.")


In [ ]:
#@title Visualize a small alignment window
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

def plot_alignment_window(windows, start=0, width=70):
    labels = list(windows)
    end = min(start + width, min(len(seq) for seq in windows.values()))
    bases = ["A", "C", "G", "T", "N", "-"]
    base_to_int = {base: i for i, base in enumerate(bases)}
    matrix = np.array([
        [base_to_int.get(base, base_to_int["N"]) for base in windows[label][start:end]]
        for label in labels
    ])

    fig, ax = plt.subplots(figsize=(11, 4.1))
    cmap = ListedColormap([BASE_COLORS[base] for base in bases])
    ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=len(bases)-1)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xticks(range(0, end - start, 10))
    ax.set_xticklabels([str(start + x) for x in range(0, end - start, 10)])
    ax.set_xlabel("aligned marker-window column")
    ax.set_title("Aligned teaching window: conserved columns stay quiet; variable columns carry the signal", loc="left", fontsize=12)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    legend_handles = [Patch(facecolor=BASE_COLORS[base], edgecolor="none", label=base) for base in bases]
    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=len(bases),
        frameon=False,
        handlelength=1.0,
        columnspacing=1.0,
    )

    for x in range(matrix.shape[1]):
        column = [windows[label][start + x] for label in labels if start + x < len(windows[label])]
        observed = {base for base in column if base in "ACGT"}
        if len(observed) > 1:
            ax.plot([x, x], [-0.48, -0.25], color="#222222", linewidth=0.7, clip_on=False)

    plt.tight_layout()
    plt.show()

plot_alignment_window(aligned_windows, ALIGNMENT_START, ALIGNMENT_WIDTH)


## 8. Compute sequence distances

Once the sequences are comparable, what is the simplest number we can calculate?

For each pair, we count how many aligned positions differ. Dividing by the number of compared positions gives a fraction different. Small values mean the two marker sequences are similar. Larger values mean they are more different.


In [ ]:
#@title Pairwise distances and closest references
def fraction_different(seq_a, seq_b):
    n = min(len(seq_a), len(seq_b))
    a = seq_a[:n]
    b = seq_b[:n]
    compared = 0
    differences = 0
    for left, right in zip(a, b):
        if left in "N-" or right in "N-":
            continue
        compared += 1
        if left != right:
            differences += 1
    return differences / compared if compared else math.nan

labels = list(aligned_windows)
dist = pd.DataFrame(index=labels, columns=labels, dtype=float)
for left in labels:
    for right in labels:
        dist.loc[left, right] = fraction_different(aligned_windows[left], aligned_windows[right])

query_labels = [record.id for record in queries]
reference_labels = [record.id for record in references]
tree_top = {}
for query in query_labels:
    ranked = dist.loc[query, reference_labels].sort_values()
    tree_top[query] = ranked.index[0]
cached_top = (
    cached_hits[cached_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
assert tree_top == cached_top, f"Computed closest hits do not match cached table: {tree_top} vs {cached_top}"

closest = (
    cached_hits[cached_hits["rank"] == 1]
    .rename(columns={
        "query_label": "query",
        "reference_label": "closest_reference",
        "fraction_different": "cached_hit_fraction_different",
        "percent_identity_teaching_window": "percent_similarity",
    })
    [["query", "closest_reference", "cached_hit_fraction_different", "percent_similarity"]]
    .copy()
)
closest["tree_distance_to_closest"] = [
    float(dist.loc[row["query"], row["closest_reference"]])
    for _, row in closest.iterrows()
]
display(closest.style.format({
    "cached_hit_fraction_different": "{:.4f}",
    "percent_similarity": "{:.2f}",
    "tree_distance_to_closest": "{:.4f}",
}))
print("Closest-reference labels match the cached hit table; identity percentages come from the cached direct-hit table.")


In [ ]:
#@title Distance matrix heatmap
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    dist,
    cmap="viridis",
    vmin=0,
    vmax=float(np.nanmax(dist.values)),
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "fraction of compared positions that differ"},
    ax=ax,
)
ax.set_title("Pairwise 16S marker distances", loc="left", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Build distance trees

How does a distance matrix become a tree?

UPGMA repeatedly joins the closest clusters. It is transparent, but it behaves like lineages changed at roughly similar rates.

Neighbor joining also starts with pairwise distances, but it is less tied to that equal-rate assumption. Comparing both methods helps us see that a tree depends on the data and the method.


In [ ]:
#@title Build UPGMA and neighbor-joining trees
def as_distance_matrix(dist_df):
    names = list(dist_df.index)
    lower = []
    for i, name in enumerate(names):
        lower.append([float(dist_df.iloc[i, j]) for j in range(i + 1)])
    return DistanceMatrix(names, lower)

dm = as_distance_matrix(dist)
constructor = DistanceTreeConstructor()
upgma_tree = constructor.upgma(dm)
nj_tree = constructor.nj(dm)
upgma_tree.rooted = True
nj_tree.rooted = False

print("UPGMA tree and neighbor-joining tree built from the same distance matrix.")


In [ ]:
#@title Plot the tree(s)
def plot_tree(tree, title):
    fig, ax = plt.subplots(figsize=(10, 5.4))
    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        show_confidence=False,
        label_func=lambda clade: clade.name if clade.name else "",
    )
    ax.set_title(title, loc="left", fontsize=12)
    ax.set_xlabel("sequence-distance-derived branch length")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", color="#eeeeee", linewidth=0.8)
    plt.tight_layout()
    plt.show()

if TREE_METHOD_TO_SHOW == "UPGMA":
    plot_tree(upgma_tree, "UPGMA distance tree")
elif TREE_METHOD_TO_SHOW == "Neighbor joining":
    plot_tree(nj_tree, "Neighbor-joining distance tree")
else:
    plot_tree(upgma_tree, "UPGMA distance tree")
    plot_tree(nj_tree, "Neighbor-joining distance tree")


## 10. Report the result carefully

What exactly did we build?

We built a distance-based gene tree from one 16S marker window. That can support a closest-reference statement. It does not prove exact species identity, and it is not a full species tree.


In [ ]:
#@title Create a student report sentence
row = closest.set_index("query").loc[QUERY_TO_REPORT]
report = (
    f"{QUERY_TO_REPORT} is closest to {row['closest_reference']} in this cached 16S marker comparison "
    f"({row['percent_similarity']:.2f}% similarity across the teaching window). "
    "Because this is one short 16S region, I can report a closest reference, not prove exact species identity or a complete species tree."
)
display(HTML(f'''
<div style='max-width: 900px; border-left: 4px solid {OKABE_ITO["bluish_green"]}; padding: 10px 14px; background: #fafafa; font-size: 15px; line-height: 1.45;'>
  <b>Careful claim:</b><br>{report}
</div>
'''))


## 11. Optional: where IQ-TREE fits

IQ-TREE is useful, but for a different teaching purpose.

UPGMA and neighbor joining are distance methods. They help students see the bridge from sequence differences to tree geometry.

IQ-TREE is a model-based maximum-likelihood tool. Use it after this lesson if you want students to compare a simple distance tree with a modern model-based tree. Do not make it the default class path for the first run.


In [ ]:
#@title Optional advanced note: MAFFT/IQ-TREE path is off by default
RUN_ADVANCED_IQTREE_PATH = False #@param {type:"boolean"}
if RUN_ADVANCED_IQTREE_PATH:
    print("Advanced path:")
    print("1. Align cleaned project reads and references with MAFFT.")
    print("2. Run IQ-TREE with model selection, for example: iqtree2 -s aligned.fasta -m MFP -B 1000 -T AUTO")
    print("3. Compare the maximum-likelihood tree with the UPGMA/NJ teaching trees.")
    print("Keep this optional so the main class run remains low-friction.")
else:
    print("Advanced IQ-TREE path skipped. The class-safe UPGMA/NJ workflow is complete.")


## 12. Replace the teaching cache later

How does this become your real soil microbiome project?

Keep the notebook structure the same, but replace the teaching FASTA and metadata with your team's cleaned 16S reads and selected database references. The critical rule stays the same: prepare the cache before class, push it to GitHub, and let Colab load known files.


In [ ]:
#@title Project replacement schema
project_schema = pd.DataFrame([
    {
        "file": "project_16s_reads.fasta",
        "required columns or fields": "FASTA id, DNA sequence",
        "example": ">TeamA_ASV_001 sample=Rhizosphere_A",
    },
    {
        "file": "project_16s_references.fasta",
        "required columns or fields": "FASTA id, accession, source database",
        "example": ">Bacillus_ref accession=NR_102783.2 source=NCBI",
    },
    {
        "file": "project_16s_metadata.csv",
        "required columns or fields": "label, role, accession, source_database, source_url, date_retrieved, taxonomy, note",
        "example": "TeamA_ASV_001, query, blank, class sample, blank, 2026-05-25, unknown, cleaned ASV",
    },
    {
        "file": "project_16s_abundance_table.csv",
        "required columns or fields": "sample_id plus one column per ASV",
        "example": "Rhizosphere_A, 128, 34, ...",
    },
])
wrapped_table(project_schema, ["file", "required columns or fields", "example"])

print("GitHub cache pattern:")
print('CACHE_BASE_URL = "https://raw.githubusercontent.com/<org>/<repo>/main/soil_16s_class_cache"')
print("Set USE_GITHUB_CACHE=True only after the cache folder is pushed.")


In [ ]:
#@title Export class outputs
output_dir = Path("soil_microbiome_16s_outputs")
output_dir.mkdir(exist_ok=True)
dist.to_csv(output_dir / "soil_16s_distance_matrix.csv")
closest.to_csv(output_dir / "soil_16s_closest_reference_report.csv", index=False)
Phylo.write(upgma_tree, output_dir / "soil_16s_upgma_tree.newick", "newick")
Phylo.write(nj_tree, output_dir / "soil_16s_neighbor_joining_tree.newick", "newick")
metadata.to_csv(output_dir / "soil_16s_metadata_used.csv", index=False)
atacama_stats.to_csv(output_dir / "atacama_top_asv_stats.csv", index=False)
atacama_alpha_stats.to_csv(output_dir / "atacama_alpha_diversity_stats.csv", index=False)
atacama_metadata.to_csv(output_dir / "atacama_sample_metadata_mini.csv", index=False)
atacama_relative.to_csv(output_dir / "atacama_relative_abundance_top12.csv", index=False)
print("Wrote outputs to:", output_dir.resolve())


## Final Think Prompts

- Which query has the closest reference in the cached set?
- Which references cluster near each other?
- Does a high 16S similarity prove exact species identity?
- Which Atacama ASV has the strongest humidity association after BH correction?
- Why are abundance q-values different from tree branch support values?
- What extra evidence would you want before making a stronger species claim?
- How would the workflow change when you replace these cached teaching reads with your team's real project reads?
